# Exercise 10.1: Stochastic simulation of ion channel gating

In the theory sections, we discussed how macroscopic currents emerge from the collective behavior of many individual, stochastically flickering ion channels. In this exercise, we will build a simulation to demonstrate this principle directly.

We will model a simple three-state channel:

$$
C \underset{k_{OC}}{\overset{k_{CO}}{\rightleftharpoons}} O \underset{k_{IO}}{\overset{k_{OI}}{\rightleftharpoons}} I
$$

where $C$ is the closed state, $O$ is the open (conducting) state, and $I$ is the inactivated state.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

## Exercise 10.1a: Building the transition matrix

The deterministic (macroscopic) dynamics of a population of these channels are governed by:

$$
\frac{\mathrm{d}\mathbf{y}}{\mathrm{d}t} = A \, \mathbf{y}
$$

where $\mathbf{y} = [C, O, I]^T$ is the state vector and $A$ is the **transition rate matrix**.

**Your task:** Complete the `transition_matrix` function below. Fill in the off-diagonal elements using the rate constants, then compute the diagonal elements so that each column sums to zero (conservation of probability).

_Hint: $A_{ij}$ (for $i \neq j$) is the rate of transition **from** state $j$ **to** state $i$.\_


In [ ]:
def transition_matrix(k_co, k_oc, k_oi, k_io):
    """Build the 3x3 transition rate matrix for the C-O-I model."""
    # State indices
    C = 0
    O = 1
    I = 2

    n = 3
    A = np.zeros((n, n))

    # TODO: Fill in the off-diagonal transition rates
    # A[to_state, from_state] = rate
    A[O, C] = ...  # C -> O
    A[C, O] = ...  # O -> C
    A[I, O] = ...  # O -> I
    A[O, I] = ...  # I -> O

    # Diagonal: each column must sum to zero
    for i in range(n):
        A[i, i] = -np.sum(A[:, i]) + A[i, i]

    return A


# Set rate constants
k_co = 5
k_oc = 10
k_oi = 1
k_io = 0

A = transition_matrix(k_co, k_oc, k_oi, k_io)
print("Transition matrix A:")
print(A)

Now solve the deterministic ODE system and plot the state probabilities over time. Start with all channels in the closed state: $\mathbf{y}_0 = [1, 0, 0]$.


In [ ]:
# Solve the deterministic ODE
def ode_system(t, y, A):
    return A @ y


# TODO: Fill in the initial conditions
y0 = np.array([..., ..., ...])  # Start in closed state
t_span = (0, 5)
dt = 0.01
t_eval = np.arange(*t_span, dt)

# TODO: Fill in the missing input to solve_ivp
sol = solve_ivp(..., ..., ..., t_eval=..., args=(...,))

# TODO: Fill in the blanks and plot the solutions
plt.plot(sol.t, ..., label="C (closed)")
plt.plot(sol.t, ..., label="O (open)")
plt.plot(sol.t, ..., label="I (inactivated)")
plt.xlabel("Time (ms)")
plt.ylabel("Probability")
plt.title("Deterministic solution")
plt.legend()
plt.show()

## Exercise 10.1b: Single-channel stochastic simulation

A single real ion channel does not follow the smooth deterministic curves above. Instead, it randomly jumps between discrete states. We can simulate this as follows:

At each time step $\Delta t$, the probability of transitioning from state $j$ to state $i$ is approximately $A_{ij} \cdot \Delta t$.

**Your task:** Complete the `advance_stochastic` function below. Given the current state, generate a random number and use the transition probabilities to decide whether the channel transitions.


In [ ]:
def advance_stochastic(state, A, dt):
    """Advance a single channel by one time step using stochastic simulation.

    Parameters
    ----------
    state : int
        Current state index (0=C, 1=O, 2=I)
    A : ndarray
        Transition rate matrix
    dt : float
        Time step

    Returns
    -------
    int
        New state index
    """
    # TODO: Get transition probabilities from current state
    # (off-diagonal elements of column 'state', multiplied by dt)
    P = ...

    # TODO: Don't allow self-transition probability
    P[state] = ...

    # Use cumulative probability to decide transition
    CP = np.cumsum(P)
    r = np.random.rand()
    for i in range(len(CP)):
        if r < CP[i]:
            state = i
            return state

    return state  # Return original state if no transition


# Simulate a single channel
dt = 0.01
t = np.arange(0, 5, dt)
s = np.zeros(len(t), dtype=int)
s[0] = 0  # Start in closed state

for i in range(len(t) - 1):
    s[i + 1] = advance_stochastic(s[i], A, dt)

plt.plot(t, s, linewidth=0.5)
plt.yticks([0, 1, 2], ["C", "O", "I"])
plt.xlabel("Time (ms)")
plt.ylabel("Channel state")
plt.title("Single channel trajectory")
plt.show()

**Questions:**

1. Run the cell above several times. Do you get the same result? Why or why not?
2. Notice that with $k_{IO} = 0$, the inactivated state is _absorbing_ — once the channel enters it, it never leaves. Is this physically realistic? What rate would you change to fix this?


## Exercise 10.1c: From single channels to macroscopic currents

Now simulate $N$ independent channels simultaneously. At each time point, the fraction of channels in the open state approximates the open probability $P_o$.

**Your task:**

1. Simulate $N = 100$ channels and plot the fraction in each state over time.
2. Overlay your result with the deterministic solution from Exercise 10.1a.
3. How large must $N$ be for the stochastic average to closely match the deterministic curve?


In [ ]:
N = 100  # Try increasing this!

S = np.zeros((len(t), N), dtype=int)
for n_ch in range(N):
    S[0, n_ch] = 0  # All channels start closed
    for i in range(len(t) - 1):
        S[i + 1, n_ch] = advance_stochastic(S[i, n_ch], A, dt)

# Calculate fraction in each state
frac_C = np.mean(S == 0, axis=1)
frac_O = np.mean(S == 1, axis=1)
frac_I = np.mean(S == 2, axis=1)

# Plot stochastic vs deterministic
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, frac_C, "C0", alpha=0.7, label=f"C (stochastic, N={N})")
ax.plot(t, frac_O, "C1", alpha=0.7, label=f"O (stochastic, N={N})")
ax.plot(t, frac_I, "C2", alpha=0.7, label=f"I (stochastic, N={N})")
ax.plot(sol.t, sol.y[0], "C0--", linewidth=2, label="C (deterministic)")
ax.plot(sol.t, sol.y[1], "C1--", linewidth=2, label="O (deterministic)")
ax.plot(sol.t, sol.y[2], "C2--", linewidth=2, label="I (deterministic)")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Fraction / Probability")
ax.set_title(f"Stochastic (N={N}) vs. Deterministic")
ax.legend(fontsize=8)
plt.show()

**Questions:**

1. How does the noise level change as you increase $N$? Can you relate this to the law of large numbers?
2. A typical mammalian ion channel has a density of ~100 channels per µm². A cardiac cell has a surface area of ~3000 µm². Approximately how many sodium channels does a real cell have? Does this explain why macroscopic currents are smooth?
